# Upgrade object_catalog to Liquid Clustering

This notebook migrates an existing `object_catalog` table to use top-level `study_uid` and `series_uid` columns with **liquid clustering** on `(study_uid)`.

**Steps:**
1. Add `study_uid` and `series_uid` columns (after `path`) if they don't exist
2. Enable Delta table properties required for liquid clustering (deletion vectors, reader/writer protocol)
3. Backfill `study_uid` and `series_uid` from the existing `meta` VARIANT column
4. Apply `CLUSTER BY (study_uid)` — liquid clustering
5. Run `OPTIMIZE` to recluster existing data
6. Verify the schema and clustering

**Prerequisite:** The table must already exist (created by `init-schema` or a prior ingest).

In [0]:
dbutils.widgets.text("table", "main.pixels_solacc.object_catalog", "Fully qualified table name")

table = dbutils.widgets.get("table")
print(f"Upgrading table: {table}")

Upgrading table: classic_pixels_catalog.tcia.object_catalog


In [0]:
existing_cols = {f.name for f in spark.table(table).schema.fields}
alter_stmts = []

if "study_uid" not in existing_cols:
    alter_stmts.append("study_uid STRING AFTER path")
if "series_uid" not in existing_cols:
    alter_stmts.append("series_uid STRING AFTER study_uid")

if alter_stmts:
    alter_sql = f"ALTER TABLE {table} ADD COLUMNS ({', '.join(alter_stmts)})"
    print(f"Executing: {alter_sql}")
    spark.sql(alter_sql)
    print("Columns added successfully.")
else:
    print("study_uid and series_uid columns already exist — skipping ADD COLUMNS.")

study_uid and series_uid columns already exist — skipping ADD COLUMNS.


In [0]:
%sql
ALTER TABLE IDENTIFIER(:table)
SET TBLPROPERTIES (
  'delta.enableDeletionVectors' = 'true',
  'delta.minReaderVersion' = '3',
  'delta.minWriterVersion' = '7'
)

In [0]:
%sql
UPDATE IDENTIFIER(:table)
SET study_uid = meta:['0020000D'].Value[0]::STRING,
    series_uid = meta:['0020000E'].Value[0]::STRING
WHERE (study_uid IS NULL OR series_uid IS NULL)
  AND meta IS NOT NULL

num_affected_rows
1416


In [0]:
%sql
ALTER TABLE IDENTIFIER(:table)
CLUSTER BY (study_uid)

In [0]:
%sql
OPTIMIZE IDENTIFIER(:table)

path,metrics
s3://classic-pixels-ext-s3-049629455384-grwjki/__unitystorage/catalogs/706424f8-584f-4bf8-bfa7-9ee985f9b0b7/tables/1cb6becf-b1ac-4037-88bc-fa18671556cf,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 4, 0, false, 0, 0, 1789496671950, 1789496680650, 8, 0, null, List(0, 0), null, 14, 14, 0, 0, List(42685137, true, false, false, null, null, null, null, 0, 0, 0, 0, 1, 3531803, 3531803, null, log, 16777216, 16777216, 1, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(164, 82, 962, 0, 0, 3521), 2, 1, 5, sizeAware, defaultSplitStrategy, null, false, 0, null, false, 0, 0, 0, null, null, null, -1.0, n/a, 0, 0, false, false, -1.0, n/a), null, 0)"
s3://classic-pixels-ext-s3-049629455384-grwjki/__unitystorage/catalogs/706424f8-584f-4bf8-bfa7-9ee985f9b0b7/tables/1cb6becf-b1ac-4037-88bc-fa18671556cf,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 4, 1, true, 0, 0, 1789496680684, 1789496682999, 8, 0, null, List(0, 0), null, 14, 14, 0, 0, List(42685137, false, false, false, null, null, null, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 8388608, 16777216, 0, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 604, 0, 0, 0), 15, 1, 1, null, null, null, false, 0, null, false, 0, 0, 0, null, null, null, -1.0, n/a, 0, 0, false, false, -1.0, n/a), null, 0)"


In [0]:
%sql
DESCRIBE TABLE EXTENDED IDENTIFIER(:table)

col_name,data_type,comment
path,string,null
study_uid,string,null
series_uid,string,null
modificationTime,timestamp,null
length,bigint,null
original_path,string,null
relative_path,string,null
local_path,string,null
extension,string,null
file_type,string,null


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN study_uid IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_study_uid,
    SUM(CASE WHEN series_uid IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_series_uid,
    SUM(CASE WHEN study_uid IS NULL THEN 1 ELSE 0 END) AS rows_missing_study_uid,
    SUM(CASE WHEN series_uid IS NULL THEN 1 ELSE 0 END) AS rows_missing_series_uid
FROM IDENTIFIER(:table)

total_rows,rows_with_study_uid,rows_with_series_uid,rows_missing_study_uid,rows_missing_series_uid
108574,107158,107158,1416,1416
